In [1]:
import os
os.chdir('/home/hrayrh/projects/incontext_spurious')
os.environ['DATA_ROOT_DIR'] = '/home/hrayrh/data'

In [2]:
import sys
sys.path.append('/home/hrayrh/projects/incontext_spurious')

In [3]:
from collections import Counter
from matplotlib import pyplot as plt
%matplotlib inline
import numpy as np
from sklearn.linear_model import LogisticRegression
from src.datamodules.datasets import SpawriousExtracted

# Inspecting SpawriousExtracted to see how well Y and S are predictable from encodings

**Dino V2 ViTB14 & ResNet-18**
* Y acc > 99%
* S acc > 99%

**Scratch ResNet-18**
* Y acc = 68%
* S acc = 95%

In [13]:
ds = SpawriousExtracted(
    root_dir='/home/hrayrh/data',
    # encoding_extractor='dinov2_vitb14',
    # encoding_extractor='resnet50',
    # encoding_extractor='resnet18',
    encoding_extractor='scratch_resnet18',
)

In [14]:
train = ds.get_subset('train')
val = ds.get_subset('val')
test = ds.get_subset('test_all_diff')
print(f'{len(train)=}, {len(val)=}, {len(test)=}')

len(train)=20300, len(val)=5044, len(test)=25344


In [15]:
train_X, examples = train[np.arange(len(train))]
print(f'{train_X.shape=}, {examples.shape=}')
train_s = examples[:, 1]
train_y = examples[:, 2]
print(f'{train_s.shape=}, {train_y.shape=}')

train_X.shape=(20300, 512), examples.shape=(20300, 3)
train_s.shape=(20300,), train_y.shape=(20300,)


In [16]:
val_X, examples = val[np.arange(len(val))]
print(f'{val_X.shape=}, {examples.shape=}')
val_s = examples[:, 1]
val_y = examples[:, 2]
print(f'{val_s.shape=}, {val_y.shape=}')

val_X.shape=(5044, 512), examples.shape=(5044, 3)
val_s.shape=(5044,), val_y.shape=(5044,)


In [17]:
test_X, examples = test[np.arange(len(test))]
print(f'{test_X.shape=}, {examples.shape=}')
test_s = examples[:, 1]
test_y = examples[:, 2]
print(f'{test_s.shape=}, {test_y.shape=}')

test_X.shape=(25344, 512), examples.shape=(25344, 3)
test_s.shape=(25344,), test_y.shape=(25344,)


## Predicting Y

In [18]:
log_reg = LogisticRegression(max_iter=10000)
log_reg.fit(train_X, train_y)
val_pred = log_reg.predict(val_X)
val_acc = np.mean(val_pred == val_y)
print(f'predicting Y, {val_acc=}')

predicting Y, val_acc=0.6841792228390167


## Predicting S

In [19]:
log_reg = LogisticRegression(max_iter=10000)
log_reg.fit(train_X, train_s)
val_pred = log_reg.predict(val_X)
val_acc = np.mean(val_pred == val_s)
print(f'predicting S, {val_acc=}') 

predicting S, val_acc=0.9547977795400476


## Predicting Y and S but on test set

In [20]:
perm = np.random.permutation(len(test_X))
test_train_size = int(0.8 * len(test_X))
test_train_X = test_X[perm[:test_train_size]]
test_train_y = test_y[perm[:test_train_size]]
test_train_s = test_s[perm[:test_train_size]]
test_val_X = test_X[perm[test_train_size:]]
test_val_y = test_y[perm[test_train_size:]]
test_val_s = test_s[perm[test_train_size:]]

log_reg = LogisticRegression(max_iter=10000)
log_reg.fit(test_train_X, test_train_y)
test_val_pred = log_reg.predict(test_val_X)
test_val_acc = np.mean(test_val_pred == test_val_y)
print(f'predicting Y, {test_val_acc=}')

log_reg = LogisticRegression(max_iter=10000)
log_reg.fit(test_train_X, test_train_s)
test_val_pred = log_reg.predict(test_val_X)
test_val_acc = np.mean(test_val_pred == test_val_s)
print(f'predicting S, {test_val_acc=}')

predicting Y, test_val_acc=0.642533043992898
predicting S, test_val_acc=0.9498914973367528


# Inspecting SpawriousEmbContextsDatasetV2 to make sure it builds correct ICL instances

In [21]:
from src.datamodules.datasets import SpawriousEmbContextsDatasetV2

In [1]:
icl_ds = SpawriousEmbContextsDatasetV2(
    root_dir='/home/hrayrh/data',
    encoding_extractor='dinov2_vitb14',
    data_length=1000000,
    context_split='train',
    query_split='train',
    context_class_size=64,
    context_group_proportions=[0.45, 0.05, 0.05, 0.45],
    query_group_proportions=[0.25, 0.25, 0.25, 0.25],
    spurious_setting='wb_erm',
    sp_token_generation_mode='opposite',
    use_context_as_intermediate_queries=False,
    permute_input_dim=False
)

NameError: name 'SpawriousEmbContextsDatasetV2' is not defined

In [23]:
input_seq, context, queries, query_indices = icl_ds[2]

In [24]:
print(f"""
    {input_seq.shape=},
    {context.shape=},
    {queries.shape=},
    {query_indices.shape=},
""")


    input_seq.shape=(384, 768),
    context.shape=(128, 3),
    queries.shape=(128, 3),
    query_indices.shape=(128,),



In [25]:
context_groups = 2 * context[:, 1] + context[:, 2]
print(Counter(context_groups))
print(58/6, 9)

Counter({3: 58, 0: 58, 2: 6, 1: 6})
9.666666666666666 9


In [26]:
query_groups = 2 * queries[:, 1] + queries[:, 2]
print(Counter(query_groups))

Counter({0: 32, 3: 32, 1: 32, 2: 32})


In [27]:
np.mean(context[:, 1] == context[:, 2])

0.90625

In [28]:
np.mean(queries[:, 1] == queries[:, 2])

0.5